<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M00/M00_Lab0_API_Key_Check.ipynb)

![Module 0 Lab 0 - API Key Check](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M00/assets/images/M00_Lab0_API_Key_Check_banner.png)


# 🔑 Lab 0: Is Your API Key Working?

**Run this before your first real lab.** In about two minutes it confirms your key is set correctly, tells you exactly which provider and models you are connected to, and points at the fix if anything is wrong. Every later lab assumes the checks on this page are green.

**Which keys do you need? This course path runs on DeepSeek.**

| Secret name (key icon in the left sidebar) | Value |
|---|---|
| `LLM_PROVIDER` | `deepseek` |
| `DEEPSEEK_API_KEY` | your key from platform.deepseek.com |
| `QWEN_API_KEY` | your Alibaba Model Studio key, needed only for the embeddings labs (M01 Lab 2, M05, M10) |

Using OpenAI instead? Then add only `OPENAI_API_KEY` and skip the other three; the toolkit defaults to OpenAI when `LLM_PROVIDER` is not set.

Enable **notebook access** on each Secret. Working in Jupyter instead of Colab? Use environment variables with the same names, or just paste the key when the setup cell prompts you; the full guide is `labs/ALTERNATIVE_PROVIDERS.md` in the course repository and the key setup page on Canvas.


In [ ]:
# === Shared lab setup: install dads5250, then import ===
# Installs the course toolkit once per runtime. Nothing else is needed:
# this lab exists only to prove your keys work.
import os
!pip install -q dads5250==0.3.0

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    DEFAULT_CHAT_MODEL,
    DEFAULT_MINI_MODEL,
    DEFAULT_EMBED_MODEL,
    LLM_PROVIDER,
)

lab_pill('M00 Lab 0: API Key Check')   # sticky banner so you always see which lab you're in


## 🔌 1. Connect

`setup_openai()` finds your key (Colab Secret, then environment variable, then a hidden prompt), connects to whichever provider your Secrets select, and makes one tiny test call. The panel below is your receipt: it names the provider and the exact models every lab will use.


In [ ]:
# ==========================================================
# 1. Connect and show the receipt
# ==========================================================
client = setup_openai()                    # loads the right key, verifies with a test call

pp({
    "provider":        LLM_PROVIDER,       # "openai" unless your LLM_PROVIDER secret says otherwise
    "chat model":      DEFAULT_CHAT_MODEL, # the reasoning model the labs use
    "mini model":      DEFAULT_MINI_MODEL, # the fast default the labs use most
    "embedding model": DEFAULT_EMBED_MODEL,
}, title="Connection receipt")


## 💬 2. Chat check

One real request and one exact expected reply. Every lab in this course needs this call to work; if this passes, all the chat, JSON mode, and tool calling labs will connect the same way.


In [ ]:
# ==========================================================
# 2. Chat check: one request, one exact expected reply
# ==========================================================
try:
    r = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        temperature=0,                              # deterministic: same reply every run
        messages=[{"role": "user", "content": "Reply with exactly: chat works"}],
    )
    pp({"chat": "PASS", "model said": r.choices[0].message.content.strip()},
       title="Chat check")
except Exception as e:
    pp({"chat": "FAIL", "error": str(e)[:200],
        "first fix": "re-check the key Secret name and that notebook access is enabled"},
       title="Chat check")


## 🧭 3. Embeddings check (needed for M01 Lab 2, M05 and M10 only)

Embeddings turn text into vectors; the RAG module lives on them. On OpenAI this uses your same key. On the DeepSeek path, embeddings ride a second key, `QWEN_API_KEY`, and the toolkit routes them automatically; without that key this check reports what is missing and every chat lab still works.


In [ ]:
# ==========================================================
# 3. Embeddings check: one sentence in, one vector out
# ==========================================================
try:
    e = client.embeddings.create(model=DEFAULT_EMBED_MODEL, input="hello world")
    pp({"embeddings": "PASS",
        "vector size": len(e.data[0].embedding),
        "model": DEFAULT_EMBED_MODEL}, title="Embeddings check")
except Exception as ex:
    pp({"embeddings": "not available",
        "why": str(ex)[:160],
        "fix": "on the DeepSeek path add the QWEN_API_KEY secret; chat labs work without it"},
       title="Embeddings check")


## 🗝️ 4. Which secrets does this runtime actually see?

When something fails, the question is always the same: is the secret really visible to the notebook? This cell answers it, showing only whether each name was found, never the value.


In [ ]:
# ==========================================================
# 4. Secret visibility: found / missing, values never shown
# ----------------------------------------------------------
# Purpose: settle the classic "but I added the secret!" debugging question.
# Defines:
#   - secret_visible() : True if a name resolves via Colab Secrets or the environment
# ==========================================================
def secret_visible(name):
    try:
        from google.colab import userdata           # Colab Secrets first, like the toolkit does
        if userdata.get(name):
            return True
    except Exception:
        pass
    return bool(os.environ.get(name))               # then the environment (Jupyter path)

pp({name: ("found" if secret_visible(name) else "missing")
    for name in ["OPENAI_API_KEY", "LLM_PROVIDER", "DEEPSEEK_API_KEY", "QWEN_API_KEY"]},
   title="Secret visibility (values are never displayed)")


## 📌 If a check failed

| Symptom | Fix |
|---|---|
| Chat FAIL, "not found" | The secret name is misspelled, or notebook access is off, or (in Jupyter) the key cell has not run in this session. |
| Chat FAIL, 401 or invalid key | An invisible space or line ending came along when you copied the key. Delete the secret, re-copy carefully, save again. |
| Everything worked yesterday, fails today | You restarted the runtime. In Colab, Secrets survive; in Jupyter, re-run your key cell first. |
| Provider shows `openai` but you set DeepSeek | The `LLM_PROVIDER` secret is misspelled or notebook access is off; section 4 above will show it as missing. |
| Embeddings "not available" on DeepSeek | Expected until you add `QWEN_API_KEY`. Only M01 Lab 2, M05 and M10 need it. |

**All green? You are done here.** Close this notebook and start Module 1.

**Difficulty: ★☆☆**
